In [1]:
import os
import sys

# 1. Environment & Hardware Optimization Overrides for AMD MI300X
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"
os.environ["HF_HOME"] = "/tmp/huggingface_cache"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "9.4.2"

if "triton" in sys.modules:
    import importlib
    import triton
    importlib.reload(triton)

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 4096
dtype = None          
load_in_4bit = True   

print("📥 Step A: Loading the core Llama-3.3-70B base model framework...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("🧠 Step B: Initializing training adapters targeting key transformer layers...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("📥 Step C: Streaming 3GPP Technical Specifications & Expert Q&A (Test Slice)...")
specs_3gpp = load_dataset("GSMA/ot-lite", "3gpp_tsg", split="test")
tele_qna   = load_dataset("GSMA/ot-lite", "teleqna", split="test")

# Robust row-by-row formatting that extracts values gracefully via dict defaults
def process_telecom_row(row):
    question = row.get("question") or row.get("instruction") or ""
    explanation = row.get("explanation") or row.get("output") or ""
    input_context = row.get("input") or ""
    
    options = row.get("options")
    options_text = f"\nOptions:\n{options}" if options else ""
    
    full_text = (
        f"### Telecom Domain Task/Question:\n{question}{options_text}\n\n"
        f"### Technical Context/Analysis:\n{input_context}\n\n"
        f"### Expert 3GPP Resolution/Explanation:\n{explanation}"
        f"{tokenizer.eos_token}"
    )
    return {"text": full_text}

print("🔄 Normalizing dataset structural schemas safely row-by-row...")
processed_specs = specs_3gpp.map(process_telecom_row, remove_columns=specs_3gpp.column_names)
processed_qna   = tele_qna.map(process_telecom_row, remove_columns=tele_qna.column_names)

# Merge the technical data files into a single training pool
phase1_dataset = concatenate_datasets([processed_specs, processed_qna])
print(f"🎯 Datasets loaded cleanly! Combined training blocks: {len(phase1_dataset)}")
print(f"📄 Sample row preview:\n\n{phase1_dataset['text'][0][:400]}...\n")

# ==========================================================
# 4. Configure Phase 1 for a deep 300-step run
# ==========================================================
training_args_phase1 = TrainingArguments(
    per_device_train_batch_size = 2,  
    gradient_accumulation_steps = 4,  
    warmup_steps = 15,
    max_steps = 300,                  # Deep step limit to force-saturate protocol knowledge
    learning_rate = 1e-4,             
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    output_dir = "./phase1_output",
    save_strategy = "no",             
    optim = "adamw_8bit",             
    lr_scheduler_type = "cosine",     
    seed = 3407,
)

trainer_phase1 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = phase1_dataset, 
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,                   # Chain rows back-to-back to maximize execution density
    args = training_args_phase1,
)

print("🏁 Launching Stage 1: Baking 3GPP Technical Specs into the adapters...")
trainer_phase1.train()

# Lock this progress permanently onto your 25GB storage disk
print("💾 Saving Stage 1 intermediate weights permanently to persistent storage...")
model.save_pretrained("stage1_3gpp_final")
tokenizer.save_pretrained("stage1_3gpp_final")
print("🎉 Stage 1 complete! Core technical specs are now fully locked onto your persistent drive.")

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 06-11 03:56:48 [__init__.py:224] Automatically detected platform rocm.
WARNING 06-11 03:56:48 [rocm.py:42] Failed to import from vllm._C with ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
WARNING 06-11 03:56:48 [rocm.py:48] Failed to import from vllm._rocm_C with ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_rocm_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/moe_utils.py:184: UserWarning: Unmatched validator: "HIP_VERSION" is required, but the tuning results does not provide it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:411.)
  torch._grouped_mm(x, w, offs=offs)
/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/moe_utils.py:184: UserWarning: Unmatched validator: "ROCM_VERSION" is provided, but pytorch is unable to consume it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:419.)
  torch._grouped_mm(x, w, offs=offs)


WARNING 06-11 03:56:48 [interface.py:178] Failed to import from vllm._C: ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
ERROR 06-11 03:56:48 [config.py:26] Failed to import Triton kernels. Please make sure your triton version is compatible.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Step A: Loading the core Llama-3.3-70B base model framework...
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,100 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 207,093,760 of 70,760,800,256 (0.29% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.701843
10,4.379555
15,3.255965
20,1.868569
25,1.483503
30,1.268538
35,1.451893
40,1.345318
45,1.332962
50,1.233050


💾 Saving Stage 1 intermediate weights permanently to persistent storage...


Unsloth: Restored added_tokens_decoder metadata in stage1_3gpp_final/tokenizer_config.json.


🎉 Stage 1 complete! Core technical specs are now fully locked onto your persistent drive.


In [1]:
import os
import sys

# 1. Environment & Hardware Optimization Overrides for AMD MI300X
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"
os.environ["HF_HOME"] = "/tmp/huggingface_cache"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "9.4.2"

if "triton" in sys.modules:
    import importlib
    import triton
    importlib.reload(triton)

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 4096
dtype = None          
load_in_4bit = True   

print("📥 Step A: Loading the core Llama-3.3-70B base model framework...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("🧠 Step B: Initializing training adapters targeting key transformer layers...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("📥 Step C: Streaming 3GPP Technical Specifications & Expert Q&A (Test Slice)...")
specs_3gpp = load_dataset("GSMA/ot-lite", "3gpp_tsg", split="test")
tele_qna   = load_dataset("GSMA/ot-lite", "teleqna", split="test")

# Robust row-by-row formatting that extracts values gracefully via dict defaults
def process_telecom_row(row):
    question = row.get("question") or row.get("instruction") or ""
    explanation = row.get("explanation") or row.get("output") or ""
    input_context = row.get("input") or ""
    
    options = row.get("options")
    options_text = f"\nOptions:\n{options}" if options else ""
    
    full_text = (
        f"### Telecom Domain Task/Question:\n{question}{options_text}\n\n"
        f"### Technical Context/Analysis:\n{input_context}\n\n"
        f"### Expert 3GPP Resolution/Explanation:\n{explanation}"
        f"{tokenizer.eos_token}"
    )
    return {"text": full_text}

print("🔄 Normalizing dataset structural schemas safely row-by-row...")
processed_specs = specs_3gpp.map(process_telecom_row, remove_columns=specs_3gpp.column_names)
processed_qna   = tele_qna.map(process_telecom_row, remove_columns=tele_qna.column_names)

# Merge the technical data files into a single training pool
phase1_dataset = concatenate_datasets([processed_specs, processed_qna])
print(f"🎯 Datasets loaded cleanly! Combined training blocks: {len(phase1_dataset)}")
print(f"📄 Sample row preview:\n\n{phase1_dataset['text'][0][:400]}...\n")

# ==========================================================
# 4. Configure Phase 1 for a deep 300-step run
# ==========================================================
training_args_phase1 = TrainingArguments(
    per_device_train_batch_size = 2,  
    gradient_accumulation_steps = 4,  
    warmup_steps = 15,
    max_steps = 300,                  # Deep step limit to force-saturate protocol knowledge
    learning_rate = 1e-4,             
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    output_dir = "./phase1_output",
    save_strategy = "no",             
    optim = "adamw_8bit",             
    lr_scheduler_type = "cosine",     
    seed = 3407,
)

trainer_phase1 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = phase1_dataset, 
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,                   # Chain rows back-to-back to maximize execution density
    args = training_args_phase1,
)

print("🏁 Launching Stage 1: Baking 3GPP Technical Specs into the adapters...")
trainer_phase1.train()

# Lock this progress permanently onto your 25GB storage disk
print("💾 Saving Stage 1 intermediate weights permanently to persistent storage...")
model.save_pretrained("stage1_3gpp_final")
tokenizer.save_pretrained("stage1_3gpp_final")
print("🎉 Stage 1 complete! Core technical specs are now fully locked onto your persistent drive.")

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 06-11 03:56:48 [__init__.py:224] Automatically detected platform rocm.
WARNING 06-11 03:56:48 [rocm.py:42] Failed to import from vllm._C with ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
WARNING 06-11 03:56:48 [rocm.py:48] Failed to import from vllm._rocm_C with ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_rocm_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/moe_utils.py:184: UserWarning: Unmatched validator: "HIP_VERSION" is required, but the tuning results does not provide it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:411.)
  torch._grouped_mm(x, w, offs=offs)
/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/moe_utils.py:184: UserWarning: Unmatched validator: "ROCM_VERSION" is provided, but pytorch is unable to consume it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:419.)
  torch._grouped_mm(x, w, offs=offs)


WARNING 06-11 03:56:48 [interface.py:178] Failed to import from vllm._C: ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
ERROR 06-11 03:56:48 [config.py:26] Failed to import Triton kernels. Please make sure your triton version is compatible.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Step A: Loading the core Llama-3.3-70B base model framework...
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,100 | Num Epochs = 3 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 207,093,760 of 70,760,800,256 (0.29% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.701843
10,4.379555
15,3.255965
20,1.868569
25,1.483503
30,1.268538
35,1.451893
40,1.345318
45,1.332962
50,1.233050


💾 Saving Stage 1 intermediate weights permanently to persistent storage...


Unsloth: Restored added_tokens_decoder metadata in stage1_3gpp_final/tokenizer_config.json.


🎉 Stage 1 complete! Core technical specs are now fully locked onto your persistent drive.


In [5]:
import os
import sys
import torch
from unsloth import FastLanguageModel

# 1. Standard environment lock
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "9.4.2"

max_seq_length = 4096

print("📥 Loading core foundational framework...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

print("🧠 Injecting and activating your custom Stage 1 3GPP adapters...")
# Force-load your specific trained weights into the model path
model.load_adapter("stage1_3gpp_final")

# Set the active model state strictly to evaluation mode
FastLanguageModel.for_inference(model)

# 2. Hard telecom protocol test prompt
prompt = """### Telecom Domain Task/Question:
A gNodeB is experiencing continuous RRC Connection Reestablishment failures with Cause Value 'Other'. Explain the protocol-level root causes according to 3GPP TS 38.331.

### Technical Context/Analysis:
"""

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

print("\n🔮 Generating Specialist Response...\n" + "="*60)
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 300, 
        use_cache = True,
        temperature = 0.3, # Low temperature keeps it strictly factual to its training data
        top_p = 0.9
    )

print(tokenizer.decode(outputs[0], skip_special_tokens = True))

📥 Mounting base layers...
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.3-70B-Instruct-bnb-4bit as a legacy tokenizer.


🧠 Mounting Stage 1 adapters...


Loading weights:   0%|          | 0/1120 [00:00<?, ?it/s]

Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔮 Forcing Specialist Generation...
1. The UE may be unable to decode DCI format 0_1 on PDCCH due to poor radio conditions.
2. The UE may not receive an RRCReconfiguration message within T311. If this timer expires before receiving the RRCReconfiguration message, the UE initiates another RRC Connection Re-establishment procedure.
3. The UE receives an RRCSetupResponse message but fails to access the Dedicated Core Network. This can happen if there are issues in the core network or when the UE’s subscription does not support dedicated core networks.
4. The UE experiences RLF and declares it at the same time as the UE receives an RRCResumeRequest message from the network. In such cases, the UE will ignore the RRCResumeRequest message and initiate an RRCConnectionRe-establishment procedure instead of responding with an RRCResumeResponse message.
5. The UE has received an RRCResumeReject message but then encounters RLF again before completing the RRC resume procedure. In this case, the UE 

In [1]:
#Because we set the learning rate slightly lower (5e-5), it will smoothly adjust the model's tone to be conversational and diagnostic without overwriting or erasing the 3GPP protocol logic it just demonstrated.
#Looking at the dataset columns natively and use a completely safe, bulletproof parsing cell that automatically finds the correct text column, regardless of its name
#It will pull the base architecture from fast local disk cache, snap Stage 1 3GPP brain onto it, map the real human text rows, and light up the execution matrix!
import os
import sys
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

# 1. Environment & Hardware Optimization Overrides
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"
os.environ["HF_HOME"] = "/tmp/huggingface_cache"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "9.4.2"

max_seq_length = 4096

print("📥 Step A: Loading the core Llama-3.3-70B base model framework...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

print("🧠 Step B: Overlaying your saved Stage 1 3GPP domain knowledge adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

# FIX: Correctly providing both the local folder path and mapping it to the default active adapter slot
model.load_adapter("stage1_3gpp_final", adapter_name="default")
print("✅ Stage 1 weights successfully injected into active memory layers.")

print("📥 Step C: Streaming Phase 2: Conversational Support Transcripts...")
phase2_dataset = load_dataset("electricsheepafrica/africa-synth-telecom-customer-service-call-transcripts-nigeria", split="train")

# Targeting the internal structural text field inside this specific dataset partition
def process_transcripts(row):
    # This dataset wraps its conversational exchanges under the 'conversation' key
    raw_conversation = row.get("conversation") or row.get("transcript") or row.get("text") or ""
    return {"text": str(raw_conversation) + tokenizer.eos_token}

print("🔄 Mapping conversational text fields row-by-row...")
phase2_dataset = phase2_dataset.map(process_transcripts, remove_columns=phase2_dataset.column_names)
print(f"🎯 Transcript data aligned! Total conversation blocks: {len(phase2_dataset)}")
print(f"📄 Text Verification Sample:\n{phase2_dataset['text'][0][:250]}...\n")

# ==========================================================
# 2. Configure Phase 2 for a deep 300-step alignment run
# ==========================================================
training_args_phase2 = TrainingArguments(
    per_device_train_batch_size = 2,  
    gradient_accumulation_steps = 4,  
    warmup_steps = 15,
    max_steps = 300,                  
    learning_rate = 5e-5,             # Protects Stage 1 weights from forgetting protocol specifics
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    output_dir = "./phase2_output",
    save_strategy = "no",             
    optim = "adamw_8bit",             
    lr_scheduler_type = "linear",     
    seed = 3407,
)

trainer_phase2 = SFTTrainer(
    model = model,                     
    tokenizer = tokenizer,
    train_dataset = phase2_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,                   
    args = training_args_phase2,
)

print("🏁 Launching Stage 2: Training adapters on conversational text dialogue...")
trainer_phase2.train()

# 3. Save the true master model permanently to your persistent disk space
print("💾 Saving the final integrated expert model to your persistent drive...")
model.save_pretrained("telco_expert_final_lora")
tokenizer.save_pretrained("telco_expert_final_lora")
print("🎉 Master Architecture Locked! Your custom 70B Telco Brain is permanently saved.")

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 06-11 05:22:21 [__init__.py:224] Automatically detected platform rocm.
WARNING 06-11 05:22:21 [rocm.py:42] Failed to import from vllm._C with ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
WARNING 06-11 05:22:21 [rocm.py:48] Failed to import from vllm._rocm_C with ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_rocm_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/moe_utils.py:184: UserWarning: Unmatched validator: "HIP_VERSION" is required, but the tuning results does not provide it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:411.)
  torch._grouped_mm(x, w, offs=offs)
/usr/local/lib/python3.12/dist-packages/unsloth_zoo/temporary_patches/moe_utils.py:184: UserWarning: Unmatched validator: "ROCM_VERSION" is provided, but pytorch is unable to consume it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:419.)
  torch._grouped_mm(x, w, offs=offs)


WARNING 06-11 05:22:21 [interface.py:178] Failed to import from vllm._C: ImportError('/usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN3c103hip28c10_hip_check_implementationEiPKcS2_ib')
ERROR 06-11 05:22:21 [config.py:26] Failed to import Triton kernels. Please make sure your triton version is compatible.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Step A: Loading the core Llama-3.3-70B base model framework...
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 207,093,760 of 70,760,800,256 (0.29% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,27.404950
10,23.374670
15,16.673987
20,15.011600
25,14.203452
30,13.880122
35,13.582660
40,13.263208
45,12.911780
50,12.478780


💾 Saving the final integrated expert model to your persistent drive...


Unsloth: Restored added_tokens_decoder metadata in telco_expert_final_lora/tokenizer_config.json.


🎉 Master Architecture Locked! Your custom 70B Telco Brain is permanently saved.


In [2]:
import torch
from unsloth import FastLanguageModel

# 1. Load the final integrated weights you just saved
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "telco_expert_final_lora",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Test Prompt: Forcing a blend of customer conversation and protocol diagnostics
prompt = """### Telecom Domain Task/Question:
A tier-1 customer support operator needs an engineering brief. A regional cluster is throwing continuous RRC Connection Reestablishment failures with Cause Value 'Other'. Explain the protocol-level root causes using T311/N310 timers from 3GPP TS 38.331.

### Expert 3GPP Resolution/Explanation:
1."""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("\n🔮 Testing Integrated Model Output...\n" + "="*60)
with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = 300, 
        use_cache = True,
        temperature = 0.3,       
        top_p = 0.9,
    )

generated_tokens = outputs[0][len(inputs["input_ids"][0]):]
print("1." + tokenizer.decode(generated_tokens, skip_special_tokens = True))

==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.3-70B-Instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔮 Testing Integrated Model Output...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppr

1.


In [2]:
import os
import sys
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

# 1. Environment & Hardware Optimization Overrides
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"
os.environ["HF_HOME"] = "/tmp/huggingface_cache"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "9.4.2"

max_seq_length = 4096

print("📥 Step A: Loading clean foundational base model architecture...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

print("🧠 Step B: Creating a completely isolated LoRA slot for Conversational Training...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

print("📥 Step C: Loading and parsing transcripts from 'customer_transcript'...")
phase2_dataset = load_dataset("electricsheepafrica/africa-synth-telecom-customer-service-call-transcripts-nigeria", split="train")

# TARGET FIX: Bind directly to the confirmed text column name
def process_real_transcripts(row):
    raw_text = row.get("customer_transcript") or ""
    operator_info = row.get("operator") or "Network"
    
    # Build a clean, rich conversational dialogue prompt block
    full_block = (
        f"### Operator/Network Carrier:\n{operator_info}\n\n"
        f"### Customer Service Interaction Transcript:\n{raw_text}"
        f"{tokenizer.eos_token}"
    )
    return {"text": full_block}

print("🔄 Mapping conversational structures...")
phase2_dataset = phase2_dataset.map(process_real_transcripts, remove_columns=phase2_dataset.column_names)

print(f"🎯 Cleaned dataset parsed! True text conversational rows: {len(phase2_dataset)}")
print(f"📄 Real Conversation Preview:\n{phase2_dataset['text'][0][:250]}...\n")

# ==========================================================
# 2. Run Phase 2 inside this clean isolated layer
# ==========================================================
training_args_phase2 = TrainingArguments(
    per_device_train_batch_size = 2,  
    gradient_accumulation_steps = 4,  
    warmup_steps = 15,
    max_steps = 150,                  # Kept tight to perfectly balance protocol logic and style
    learning_rate = 1e-4,             
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    output_dir = "./phase2_output",
    save_strategy = "no",             
    optim = "adamw_8bit",             
    lr_scheduler_type = "cosine",     
    seed = 3407,
)

trainer_phase2 = SFTTrainer(
    model = model,                     
    tokenizer = tokenizer,
    train_dataset = phase2_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,                   
    args = training_args_phase2,
)

print("🏁 Launching Clean Phase 2 Training Loop...")
trainer_phase2.train()

# Save these newly minted conversational adapters to their own folder
model.save_pretrained("stage2_conversational_final")
print("✅ Stage 2 conversational adapters saved independently.")

# ==========================================================
# 3. The Grand Master Merge: Blending Phase 1 and Phase 2 Cleanly
# ==========================================================
print("\n🧬 Initiating Grand Master Multi-Adapter Integration...")
del model, trainer_phase2
import gc
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

print("🔮 Stitching Stage 1 (3GPP) and Stage 2 (Conversational) into a unified brain...")
model.load_adapter("stage1_3gpp_final", adapter_name="3gpp_protocol")
model.load_adapter("stage2_conversational_final", adapter_name="conversational_style")

# Set active weights to combine both channels (50% domain facts + 50% conversational phrasing)
model.set_adapter(["3gpp_protocol", "conversational_style"])

print("💾 Writing unified combined master adapters permanently to storage drive...")
model.save_pretrained("telco_expert_master_integrated_lora")
tokenizer.save_pretrained("telco_expert_master_integrated_lora")
print("🎉 Success! Your true dual-knowledge 70B Telco Brain is live and permanently protected.")

📥 Step A: Loading clean foundational base model architecture...
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.3-70B-Instruct-bnb-4bit as a legacy tokenizer.


🧠 Step B: Creating a completely isolated LoRA slot for Conversational Training...


Unsloth 2026.6.2 patched 80 layers with 80 QKV layers, 80 O layers and 80 MLP layers.


...

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Loading weights:   0%|          | 0/1120 [00:00<?, ?it/s]

LlamaForCausalLM LOAD REPORT from: stage2_conversational_final
Key                                                                | Status  | 
-------------------------------------------------------------------+---------+-
model.layers.{0...79}.mlp.gate_proj.lora_B.3gpp_protocol.weight    | MISSING | 
model.layers.{0...79}.self_attn.v_proj.lora_B.3gpp_protocol.weight | MISSING | 
model.layers.{0...79}.mlp.up_proj.lora_B.3gpp_protocol.weight      | MISSING | 
model.layers.{0...79}.self_attn.v_proj.lora_A.3gpp_protocol.weight | MISSING | 
model.layers.{0...79}.self_attn.o_proj.lora_B.3gpp_protocol.weight | MISSING | 
model.layers.{0...79}.mlp.gate_proj.lora_A.3gpp_protocol.weight    | MISSING | 
model.layers.{0...79}.mlp.up_proj.lora_A.3gpp_protocol.weight      | MISSING | 
model.layers.{0...79}.mlp.down_proj.lora_A.3gpp_protocol.weight    | MISSING | 
model.layers.{0...79}.self_attn.k_proj.lora_A.3gpp_protocol.weight | MISSING | 
model.layers.{0...79}.self_attn.o_proj.lora_A.3gpp_protoc

💾 Writing unified combined master adapters permanently to storage drive...


ValueError: Multiple active adapters detected, saving multiple active adapters is not supported yet. You can save adapters separately one by one by iteratively calling `model.set_adapter(adapter_name)` then `model.save_pretrained(...)`

In [7]:
import os
import sys
import torch
from unsloth import FastLanguageModel
from safetensors.torch import load_file

# 1. Environment Locks
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "9.4.2"

max_seq_length = 4096

print("📥 Step 1: Initializing a clean base model with active training adapters...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

print("\n💾 Step 2: Fetching weight dictionaries from local disk files...")
# Safetensors Fallback Check Logic
path_3gpp = "stage1_3gpp_final/adapter_model.safetensors" if os.path.exists("stage1_3gpp_final/adapter_model.safetensors") else "stage1_3gpp_final/adapter_model.bin"
path_conv = "stage2_conversational_final/adapter_model.safetensors" if os.path.exists("stage2_conversational_final/adapter_model.safetensors") else "stage2_conversational_final/adapter_model.bin"

print(f"📂 Loading Stage 1 From: {path_3gpp}")
print(f"📂 Loading Stage 2 From: {path_conv}")

# Load the dictionaries based on their format type
state_dict_3gpp = load_file(path_3gpp) if path_3gpp.endswith(".safetensors") else torch.load(path_3gpp, map_location="cpu")
state_dict_conv = load_file(path_conv) if path_conv.endswith(".safetensors") else torch.load(path_conv, map_location="cpu")

print("🧬 Step 3: Performing manual element-wise matrix interpolation (50/50 balance)...")
merged_state_dict = {}
blended_count = 0

for key in state_dict_3gpp.keys():
    if key in state_dict_conv:
        weight_3gpp = state_dict_3gpp[key].float()
        weight_conv = state_dict_conv[key].float()
        
        # Calculate the mathematical mid-point matrix
        blended_weight = (0.5 * weight_3gpp) + (0.5 * weight_conv)
        
        # Cast precision cleanly back to match the foundational architecture
        merged_state_dict[key] = blended_weight.to(state_dict_3gpp[key].dtype)
        blended_count += 1
    else:
        merged_state_dict[key] = state_dict_3gpp[key]

print(f"✅ Successfully blended {blended_count} distinct LoRA weight matrices!")

print("\n🧠 Step 4: Injecting the unified blended state dictionary into the model layers...")
model.load_state_dict(merged_state_dict, strict=False)

print("💾 Step 5: Committing the true integrated model permanently to storage drive...")
model.save_pretrained("telco_expert_master_integrated_lora")
tokenizer.save_pretrained("telco_expert_master_integrated_lora")

print("\n🎉 Success! Your true dual-knowledge 70B Telco Brain is live and permanently protected.")

[asyncio|ERROR]Task was destroyed but it is pending!
task: <Task pending name='Task-123' coro=<_async_in_context.<locals>.run_in_context() done, defined at /usr/local/lib/python3.12/dist-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-124' coro=<Kernel.shell_main() running at /usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /usr/local/lib/python3.12/dist-packages/zmq/eventloop/zmqstream.py:563]>
/usr/local/lib/python3.12/dist-packages/h11/_headers.py:247: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  if found_name == name:
[asyncio|ERROR]Task was destroyed but it is pending!
task: <Task pending name='Task-124' coro=<Kernel.shell_main() running at /usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


📥 Step 1: Initializing a clean base model with active training adapters...
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.3-70B-Instruct-bnb-4bit as a legacy tokenizer.



💾 Step 2: Fetching weight dictionaries from local disk files...
📂 Loading Stage 1 From: stage1_3gpp_final/adapter_model.safetensors
📂 Loading Stage 2 From: stage2_conversational_final/adapter_model.safetensors
🧬 Step 3: Performing manual element-wise matrix interpolation (50/50 balance)...
✅ Successfully blended 1120 distinct LoRA weight matrices!

🧠 Step 4: Injecting the unified blended state dictionary into the model layers...
💾 Step 5: Committing the true integrated model permanently to storage drive...


Unsloth: Restored added_tokens_decoder metadata in telco_expert_master_integrated_lora/tokenizer_config.json.



🎉 Success! Your true dual-knowledge 70B Telco Brain is live and permanently protected.


In [11]:
# this test prompt simulates a complex real-world scenario—a customer care interaction that requires deep technical network diagnosis.
import torch
from unsloth import FastLanguageModel

# 1. Load your brand-new, unified master model
print("📥 Mounting the Unified Master Telco Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "telco_expert_master_integrated_lora",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Challenge Prompt: Tests conversational tone mixed with protocol-level accuracy
prompt = """### Operator/Network Carrier:
MTN

### Customer Service Interaction Transcript:
AGENT: Thank you for calling network support. How can I assist you today?
CUSTOMER: Hello, my phone keeps dropping connection completely when I move between cell tower sectors in the central district. It just shows 'No Service' for a minute before resetting. Please check what is wrong.
AGENT: Let me pull up the gNodeB cluster logs for your current sector location.
### Technical Engineering Brief & Protocol Root Cause Analysis:
1."""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("\n🔮 Generating Integrated Expert Resolution...\n" + "="*60)
with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = 350, 
        use_cache = True,
        temperature = 0.4,       
        top_p = 0.9,
        repetition_penalty = 1.15
    )

generated_tokens = outputs[0][len(inputs["input_ids"][0]):]
print("1." + tokenizer.decode(generated_tokens, skip_special_tokens = True))

📥 Mounting the Unified Master Telco Model...
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.3-70B-Instruct-bnb-4bit as a legacy tokenizer.



🔮 Generating Integrated Expert Resolution...


Both `max_new_tokens` (=350) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Ignoring clean_up_tokeni

1. **Initial Assessment**: The customer's issue of intermittent loss of service (LOS) and inability to maintain connectivity while moving through different cell towers suggests potential problems with either the handover process or issues within the radio access network itself.

2. **Data Collection**:
    - Collect recent performance monitoring data from the affected area, focusing on key performance indicators such as Handover Success Rate (HSR), Drop Call Rate (DCR), and Signal-to-Noise Ratio (SNR).
    - Review configuration parameters related to handovers, including neighbor lists, hysteresis values, and time-to-trigger settings.

3. **Analysis**:
    - Analyze the collected data to identify patterns or anomalies that could indicate specific root causes. For example, low HSR might suggest issues with neighbor list configurations or inadequate overlap between cells.
    - Use drive test data if available to pinpoint areas where signal strength drops significantly during handovers.


In [2]:
import os
import sys
import torch
import warnings

# 1. Complete logging silencing
warnings.filterwarnings("ignore")
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "9.4.2"

from unsloth import FastLanguageModel

print("📥 Mounting the Unified Master Telco Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "telco_expert_master_integrated_lora",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Improvised Prompt (Slight structure shift to entice continuation)
prompt = """### Operator/Network Carrier:
MTN (5G NSA/SA Deployment)

### Customer Service Interaction Transcript:
AGENT: Thank you for calling network support. How can I assist you today?
CUSTOMER: Hello, my phone keeps dropping connection completely when I move between cell tower sectors in the central district. It just shows 'No Service' for a minute before resetting. Please check what is wrong.
AGENT: Let me pull up the gNodeB cluster logs for your current sector location.

### Low-Level Protocol Root Cause Trace (3GPP TS 38.331 Metrics):
1. **Mobility and Handover Execution Failure Analysis**:
   - The symptoms point to a failure during inter-cell or inter-gNodeB handover execution. Specifically, when the UE crosses sector boundaries, it triggers an `RRCReconfiguration` message containing `ReconfigurationWithSync`.
   - """

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("\n🔮 Forcing Advanced Deep-Dive Resolution (700 Tokens Max)...")
print("="*65)

# 3. Use generation parameters that restrict instant EOS closure
with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = 700, 
        use_cache = True,
        temperature = 0.5,       # Increased slightly to break deterministic EOS loops
        top_p = 0.85,
        repetition_penalty = 1.2, # Discourages repeating immediate stop signatures
        eos_token_id = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")] 
    )

# Cleanly display text by stripping out the raw prompt tokens
generated_tokens = outputs[0][len(inputs["input_ids"][0]):]
new_text = tokenizer.decode(generated_tokens, skip_special_tokens = True)

print("### Low-Level Protocol Root Cause Trace (3GPP TS 38.331 Metrics):\n1. **Mobility and Handover Execution Failure Analysis**:\n   - The symptoms point to a failure during inter-cell or inter-gNodeB handover execution. Specifically, when the UE crosses sector boundaries, it triggers an `RRCReconfiguration` message containing `ReconfigurationWithSync`.\n   - " + new_text.strip())

📥 Mounting the Unified Master Telco Model...
==((====))==  Unsloth 2026.6.2: Fast Llama patching. Transformers: 5.11.0. vLLM: 0.11.0rc2.dev424+g045b396d0.rocm700.
   \\   /|    AMD gfx942 GPU. Num GPUs = 1. Max memory: 191.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.0.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.3-70B-Instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=700) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔮 Forcing Advanced Deep-Dive Resolution (700 Tokens Max)...
### Low-Level Protocol Root Cause Trace (3GPP TS 38.331 Metrics):
1. **Mobility and Handover Execution Failure Analysis**:
   - The symptoms point to a failure during inter-cell or inter-gNodeB handover execution. Specifically, when the UE crosses sector boundaries, it triggers an `RRCReconfiguration` message containing `ReconfigurationWithSync`.
   - 3GPP metrics indicate that there's an issue with synchronization timing (`syncInfo`) leading to failed reconfigurations due to mismatched system frame numbers (`SFN`) at the target cell site.

2. **Xn/X2 Interface Congestion Impact Assessment**:
   - Xn interface latency spikes are observed coinciding with increased traffic load on neighboring cells, causing delays in transferring necessary signaling information required for successful handovers.
   
3. **gNB-CU/DU Synchronization Verification**:
   - Logs from both Central Unit (CU) and Distributed Unit (DU) of the serving gNod